<a href="https://colab.research.google.com/github/RagaSandhiya05/SmartText-Predictor/blob/main/Next_Word_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Install & Import Libraries**

In [4]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding , LSTM , Dense

**Load Dataset**

In [5]:
path = "1661-0.txt"
with open(path , "r" , encoding = "utf-8") as file :
  text = file.read().lower()
print("Corpus Length : " , len(text))

Corpus Length :  581888


**Tokenize Text**

In [6]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1
print("Vocabulary Size : " , total_words)

Vocabulary Size :  8932


**Create Sequences**

In [8]:
input_sequences = []
for line in text.split(".") :
  token_list = tokenizer.texts_to_sequences([line])[0]
  for i in range(1 , len(token_list)) :
    n_gram_sequence = token_list[:i + 1]
    input_sequences.append(n_gram_sequence)
print("Total Sequences : " , len(input_sequences))

Total Sequences :  104820


**Padding**

In [9]:
max_sequence_len = max(
    [len(seq) for seq in input_sequences]
)
input_sequences = np.array(
    pad_sequences(
        input_sequences ,
        maxlen = max_sequence_len ,
        padding = "pre"
    )
)

**Create X and y**

In [10]:
X = input_sequences[: , :-1]
y = input_sequences[: , -1]
print(X.shape)
print(y.shape)

(104820, 104)
(104820,)


**Build LSTM Model**

In [15]:
model = Sequential([
    Embedding(
        input_dim = total_words ,
        output_dim = 100
    ) ,
    LSTM(150) ,
    Dense(
        total_words ,
        activation = "softmax"
    )
])

**Compile Model**

In [16]:
model.compile(
    loss = "sparse_categorical_crossentropy" ,
    optimizer = "adam" ,
    metrics = ["accuracy"]
)
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

**Train Model**

In [17]:
history = model.fit(
    X ,
    y ,
    epochs = 20 ,
    batch_size = 128 ,
    validation_split = 0.1
)

Epoch 1/20
738/738 ━━━━━━━━━━━━━━━━━━━━ 331s 445ms/step - accuracy: 0.0604 - loss: 6.5901 - val_accuracy: 0.0748 - val_loss: 6.7829
Epoch 2/20
738/738 ━━━━━━━━━━━━━━━━━━━━ 324s 438ms/step - accuracy: 0.0961 - loss: 5.9212 - val_accuracy: 0.1048 - val_loss: 6.5479
Epoch 3/20
738/738 ━━━━━━━━━━━━━━━━━━━━ 324s 439ms/step - accuracy: 0.1262 - loss: 5.5405 - val_accuracy: 0.1201 - val_loss: 6.4936
Epoch 4/20
738/738 ━━━━━━━━━━━━━━━━━━━━ 327s 443ms/step - accuracy: 0.1415 - loss: 5.2915 - val_accuracy: 0.1233 - val_loss: 6.4810
Epoch 5/20
738/738 ━━━━━━━━━━━━━━━━━━━━ 329s 446ms/step - accuracy: 0.1521 - loss: 5.0855 - val_accuracy: 0.1276 - val_loss: 6.5193
Epoch 6/20
738/738 ━━━━━━━━━━━━━━━━━━━━ 377s 439ms/step - accuracy: 0.1619 - loss: 4.9050 - val_accuracy: 0.1341 - val_loss: 6.5434
Epoch 7/20
738/738 ━━━━━━━━━━━━━━━━━━━━ 394s 456ms/step - accuracy: 0.1725 - loss: 4.7406 - val_accuracy: 0.1337 - val_loss: 6.6179
Epoch 8/20
738/738 ━━━━━━━━━━━━━━━━━━━━ 334s 452ms/step - accuracy: 0.1804 -

**Save Model**

In [18]:
model.save("next_word_prediction_model.keras")

**Predict Next Word**

In [19]:
def predict_next_word(seed_text) :
  token_list = tokenizer.texts_to_sequences(
      [seed_text]
  )[0]
  token_list = pad_sequences(
      [token_list] ,
      maxlen = max_sequence_len - 1 ,
      padding = "pre"
  )
  predicted = model.predict(
      token_list ,
      verbose = 0
  )
  predicted_word_id = np.argmax(predicted)
  for word , index in tokenizer.word_index.items() :
    if index == predicted_word_id :
      return word
  return ""

**Test Model**

In [75]:
seed_text = "the adventures of sherlock"
print(
    "Next Word : " ,
    predict_next_word(seed_text)
)

Next Word :  holmes


**Predict Multiple Words**

In [22]:
def generate_text(seed_text , next_words = 10) :
  for _ in range(next_words) :
    next_word = predict_next_word(seed_text)
    seed_text += " " + next_word
  return seed_text

**Test:**

In [80]:
print(
    generate_text(
        "the adventures of sherlock" ,
        10
    )
)

the adventures of sherlock holmes sat in the door and ushered in a hearty
